# Create Parquet Files #
This program converts the `.csv` and `.xml` files from Statistics Canada into `.parquet` files. These `.parquet` files represent a transitional state. The second [jupyter notebook](backend/cleanup_parquet.ipynb), should be run after this one to create the final `.parquet` files used by the actual program.

In [ ]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import subprocess
import tempfile
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, lit, create_map
import shutil
import xml.etree.ElementTree as ET
from itertools import chain
from pathlib import Path

In [ ]:
# Define schemas according to native Statistics Canada layouts
CENSUS_CONFIGS = {
    2021: {
        'usecols': [
            'CENSUS_YEAR', 'DGUID', 'GEO_LEVEL', 'GEO_NAME',
            'CHARACTERISTIC_ID', 'CHARACTERISTIC_NAME',
            'C1_COUNT_TOTAL', 'C2_COUNT_MEN+', 'C3_COUNT_WOMEN+',
            'C10_RATE_TOTAL', 'C11_RATE_MEN+', 'C12_RATE_WOMEN+'
        ],
        'dtypes': {
            'CENSUS_YEAR': np.int64,
            'DGUID': str,
            'GEO_LEVEL': str,
            'GEO_NAME': str,
            'CHARACTERISTIC_ID': np.int64,
            'CHARACTERISTIC_NAME': str,
            'C1_COUNT_TOTAL': np.double,
            'C2_COUNT_MEN+': np.double,
            'C3_COUNT_WOMEN+': np.double,
            'C10_RATE_TOTAL': np.double,
            'C11_RATE_MEN+': np.double,
            'C12_RATE_WOMEN+': np.double,
        },
        'zips': [
            './census_data/98-401-X2021006_eng_CSV.zip',
            './census_data/98-401-X2021010_eng_CSV.zip',
            './census_data/98-401-X2021029_eng_CSV.zip',
            './census_data/98-401-X2021012_eng_CSV.zip'
        ]
    },
    2016: {
        'names': [
            'CENSUS_YEAR', 
            'GEO_CODE',
            'GEO_LEVEL',
            'GEO_NAME',
            'GNR',
            'GNR_LF',
            'DATA_QUALITY_FLAG',
            'ALT_GEO_CODE',
            'CHARACTERISTIC_NAME',
            'CHARACTERISTIC_ID',
            'NOTES',
            'COUNT_TOTAL',
            'COUNT_MALE',
            'COUNT_FEMALE'
        ],
        'cols': [
            'CENSUS_YEAR',
            'GEO_CODE',
            'GEO_LEVEL',
            'GEO_NAME',
            'CHARACTERISTIC_NAME',
            'CHARACTERISTIC_ID',
            'COUNT_TOTAL',
            'COUNT_MALE',
            'COUNT_FEMALE'            

        ],
        'dtypes': {
            'CENSUS_YEAR': np.int64,
            'GEO_CODE': str,
            'GEO_LEVEL': str,
            'GEO_NAME': str,
            'GNR': np.double,
            'GNR_LF': np.double,
            'DATA_QUALITY_FLAG': str,
            'ALT_GEO_CODE': str,
            'CHARACTERISTIC_NAME': str,
            'CHARACTERISTIC_ID': np.int64,
            'COUNT_TOTAL': np.double,
            'COUNT_MALE': np.double,
            'COUNT_FEMALE': np.double
        },
        'zips': [
            './census_data/98-401-X2016044_eng_CSV.zip',
            './census_data/98-401-X2016045_eng_CSV.zip',
            './census_data/98-401-X2016050_eng_CSV.zip'
        ]
    },
    2011: { # this is for the 2011 NHS data
        'zips': [
            './census_data/99-004-XWE2011001-101_CSV.zip',
            './census_data/99-004-XWE2011001-301_CSV.zip',
            './census_data/99-004-XWE2011001-401_CSV.zip',
            './census_data/99-004-XWE2011001-501_CSV.zip',
            './census_data/99-004-XWE2011001-701_CSV.zip'
        ]
    },
    2010: { # use 2010 for 2011 census data (not NHS)
        'zips': [
            './census_data/98-316-XWE2011001-101_CSV.zip',
            './census_data/98-316-XWE2011001-301_CSV.zip',
            './census_data/98-316-XWE2011001-401_CSV.zip',
            './census_data/98-316-XWE2011001-501_CSV.zip',
            './census_data/98-316-XWE2011001-701_CSV.zip',
            './census_data/98-316-XWE2011001-1501_CSV.zip',
        ]
    },
    2006: {
        'zips': [
            './census_data/94-581-XCB2006002.ZIP',
            './census_data/94-581-XCB2006005.ZIP',
            './census_data/94-581-XCB2006007.ZIP',
        ],
        'xml_mappings': 'mappings/xml_mappings/2006_xml_id_mappings.csv'
    },
    2001: {
        'zips': [
            './census_data/95F0495XCB2001002.ZIP',
            './census_data/95F0495XCB2001005.ZIP',
            './census_data/95F0495XCB2001007.ZIP'
        ],
        'xml_mappings': 'mappings/xml_mappings/2001_xml_id_mappings.csv'
    },
    1996: {
        'zips': [
            './census_data/95F0180XDB96001.ZIP',
            './census_data/95F0181XDB96001.ZIP',
            './census_data/95F0183XDB.ZIP',
            './census_data/95F0185XDB96001.ZIP'
        ],
        'xml_mappings': 'mappings/xml_mappings/1996_xml_id_mappings.csv'
    },
    1991: {
        'zips': [
            './census_data/1991_Census_Data.zip'
        ],
        'csv_mappings': 'mappings/csv_mappings/1991_csv_id_mappings.csv'
    },
    1986: {
        'zips': [
            './census_data/1986_Census_Data.zip'
        ]
    },
    1981: {
        'zips': [
            './census_data/1981_Census_Data.zip'
        ],
        'csv_mappings': 'mappings/csv_mappings/1981_csv_id_mappings.csv'
    }
}

output_dir = "temp_parquet_data"
os.makedirs(output_dir, exist_ok=True)

second_out_dir = "parquet_data"
os.makedirs(second_out_dir, exist_ok=True)


# Preprocess 2021 Data #

In [ ]:
def process_year_data(year_config, output_path):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith('.csv'):
                    continue

                print(f"Processing {file}")

                with z.open(file) as f:
                    for chunk in pd.read_csv(
                        f,
                        encoding='latin1',
                        low_memory=False,
                        usecols=year_config['usecols'],
                        dtype=year_config['dtypes'],
                        chunksize=100_000
                    ):

                        table = pa.Table.from_pandas(chunk, preserve_index=False)

                        if writer is None:
                            writer = pq.ParquetWriter(
                                output_path,
                                table.schema,
                                compression='snappy'
                            )

                        writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

In [ ]:
out_file = os.path.join(
    output_dir,
    "2021_raw_census_data.parquet"
)

process_year_data(CENSUS_CONFIGS[2021], out_file)

# Preprocess 2016 Data #

In [ ]:
# Regex patterns for 2016 dynamic characteristic columns
CHAR_NAME_RE = re.compile(r'^DIM:\s*Profile of.*$', re.I)
CHAR_ID_RE = re.compile(r'^Member ID:\s*Profile of.*$', re.I)

def process_2016_data(year_config, output_path):
    writer = None

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue        

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith('.csv'):
                    continue

                if file == 'Geo_starting_row_CSV.csv':
                    continue

                print(f"Processing {file}")
                with z.open(file) as f:
                    for chunk in pd.read_csv(
                        f,
                        encoding='utf-8-sig',
                        low_memory=False,
                        header=0,
                        names=year_config['names'],
                        dtype=year_config['dtypes'],
                        na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                        chunksize=100_000
                    ):
                        chunk = chunk[year_config['cols']]

                        table = pa.Table.from_pandas(chunk, preserve_index=False)

                        if writer is None:
                            writer = pq.ParquetWriter(
                                output_path,
                                table.schema,
                                compression='snappy'
                            )

                        writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")


In [ ]:
out_file = os.path.join(
    output_dir,
    "2016_raw_census_data.parquet"
)

process_2016_data(CENSUS_CONFIGS[2016], out_file)

# Preprocess 2011 Data #

In [ ]:
"""
Create Characteristic IDs for 2011 Data
"""

path_to_characteristics = Path("./characteristics/2011_characteristics.csv")


census_csv = "./2011_sample/Download_06_37_02.csv"
nhs_csv = "./2011_sample/Download_06_37_45.csv"

characteristics = []
characteristic_id = 1

# 2011 Census
df = pd.read_csv(
    census_csv,
    header=1,
    usecols=['Characteristics']
)

for value in df.iloc[:, 0]:
    characteristics.append({
        "CHARACTERISTIC_ID": characteristic_id,
        "CHARACTERISTIC_NAME": value,
        "DATASET": "Census"
    })
    characteristic_id += 1

census_2011_len = len(df)

# 2011 NHS
df = pd.read_csv(
    nhs_csv,
    header=1,
    usecols=['Characteristics']
)

for value in df.iloc[:, 0]:
    characteristics.append({
        "CHARACTERISTIC_ID": characteristic_id,
        "CHARACTERISTIC_NAME": value,
        "DATASET": "NHS"
    })
    characteristic_id += 1

characteristics_df = pd.DataFrame(characteristics)

path_to_characteristics.parent.mkdir(
    parents=True,
    exist_ok=True
)

characteristics_df.to_csv(
    path_to_characteristics,
    index=False
)

nhs_2011_len = len(df)

print(f"Saved {len(characteristics_df)} characteristics to {path_to_characteristics}")

In [ ]:
def process_2011_data(year_config, output_path, isNHS):
    writer = None
    if isNHS:
        seq_len = nhs_2011_len
        characteristic_offset = census_2011_len
    else:
        seq_len = census_2011_len
        characteristic_offset = 0

    for zip_path in year_config['zips']:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with tempfile.TemporaryDirectory() as temp_dir:
            subprocess.run(
                ["7z", "x", "-y", f"-o{temp_dir}", zip_path],
                check=True,
                stdout=subprocess.DEVNULL
            )

            for root, _, files in os.walk(temp_dir):
                for file in files:

                    if not file.endswith('.csv'):
                        continue

                    if file.endswith('DQ.csv') or file.endswith('Metadata.csv'):
                        continue

                    file_path = os.path.join(root, file)

                    print(f"Processing {file}")

                    with open(file_path, "rb") as f:
                        # Find where the metadata starts
                        metadata_start_index = None

                        for i, line in enumerate(f):
                            if line.decode("latin1").startswith("Notes:"):
                                metadata_start_index = i
                                break

                        f.seek(0)

                        # sometimes, there is an extra row at the top
                        df = pd.read_csv(f, nrows=1, header=None, encoding='latin1')
                        top_left_value = df.iat[0, 0]

                        if top_left_value != "Geo_Code":
                            # there is a useless row at the top
                            header = 1
                        else:
                            header = 0

                        f.seek(0)
                        """
                        The 2011 census data has different number of columns depending on the geographic level.
                        After analyzing the .csv file structure, I noticed that the name of the geography usually
                        appears one column to the left of the 'Topic' column. 
                        """

                        all_columns = pd.read_csv(
                            f,
                            header=header,
                            nrows=0,
                            encoding='latin1'
                        ).columns.tolist()


                        topic_col_idx = all_columns.index('Topic')

                        geo_index = topic_col_idx - 1
                        has_type_col_flag = False
                        has_gnr_col_flag = False

                        if all_columns[geo_index] == "GNR":
                            has_gnr_col_flag = True
                            # remove the GNR column
                            geo_index -= 1

                        if all_columns[geo_index].lower().endswith("type"):
                            has_type_col_flag = True
                            # type column. No need to keep
                            geo_index -= 1

                        if geo_index > 1:
                            columns_to_skip = [1, geo_index - 1]
                        else:
                            columns_to_skip = []

                        columns_to_keep = [
                            i for i in range(len(all_columns))
                            if i not in columns_to_skip
                        ]

                        # do not need to keep the type column
                        if has_type_col_flag and has_gnr_col_flag:
                            columns_to_keep.remove(topic_col_idx - 1)
                            columns_to_keep.remove(topic_col_idx - 2)

                        elif has_type_col_flag:
                            columns_to_keep.remove(topic_col_idx - 1)

                        elif has_gnr_col_flag:
                            columns_to_keep.remove(topic_col_idx - 1)


                        # do not need topic, note, or flags
                        columns_to_keep.remove(topic_col_idx)
                        # note is two to the right of topic
                        columns_to_keep.remove(topic_col_idx + 2)
                        # flag total is two to the right of topic
                        columns_to_keep.remove(topic_col_idx + 4)
                        # flag male is two to the right of flag total
                        columns_to_keep.remove(topic_col_idx + 6)
                        # flag female is two to the right of male total
                        columns_to_keep.remove(topic_col_idx + 8)
                        # if you were offended that male flag came before female flag, 
                        # you need to calm down.

                        f.seek(0)

                        read_args = {
                            "encoding": "latin1",
                            "low_memory": False,
                            "header": header,
                            "usecols": columns_to_keep,
                            "na_values": ['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                            "chunksize": 10_000
                        }

                        if metadata_start_index is not None:
                            # Header is line 1, data starts at line 2.
                            read_args["nrows"] = metadata_start_index - (1 + header)

                        row_offset = 0

                        for chunk in pd.read_csv(
                            f,
                            **read_args
                        ):

                            # ensure consistent column names
                            chunk.columns = [
                                "GEO_CODE",
                                "GEO_NAME",
                                "CHARACTERISTIC_NAME",
                                "COUNT_TOTAL",
                                "COUNT_MALE",
                                "COUNT_FEMALE"
                            ]

                            chunk = chunk.astype({
                                "GEO_CODE": str,
                                "GEO_NAME": str,
                                "CHARACTERISTIC_NAME": str,
                                "COUNT_TOTAL": np.double,
                                "COUNT_MALE": np.double,
                                "COUNT_FEMALE": np.double
                            })

                            # Cyclic characteristic assignment starting from 1 to seq_len
                            chunk_indices = np.arange(row_offset, row_offset + len(chunk))
                            chunk["CHARACTERISTIC_ID"] = (chunk_indices % seq_len) + 1 + characteristic_offset
                            row_offset += len(chunk)

                            schema = pa.schema([
                                ("GEO_CODE", pa.string()),
                                ("GEO_NAME", pa.string()),
                                ("CHARACTERISTIC_ID", pa.int64()),
                                ("CHARACTERISTIC_NAME", pa.string()),
                                ("COUNT_TOTAL", pa.float64()),
                                ("COUNT_MALE", pa.float64()),
                                ("COUNT_FEMALE", pa.float64()),
                            ])

                            table = pa.Table.from_pandas(
                                chunk,
                                schema=schema,
                                preserve_index=False
                            )

                            if writer is None:
                                writer = pq.ParquetWriter(
                                    output_path,
                                    schema,
                                    compression="snappy"
                                )


                            writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

In [ ]:
"""
Since the 2011 NHS data for dissemination areas came in a completely different format,
I have decided to treat it as a completely different dataset. It was nearly impossible
to merge this with the other NHS datasets due to their extreme differences.
"""

def process_2011_da_data(output_path):
    writer = None
    characteristic_id = census_2011_len + 1  # 473

    with zipfile.ZipFile('census_data/2011_NHS_DA.zip') as z:
        for file in z.namelist():
            print(f"Processing {file}")

            # Reopen the file for each operation because read_csv consumes it
            with z.open(file) as f:
                all_columns = pd.read_csv(
                    f,
                    nrows=0,
                    encoding='latin1',
                    na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL'],
                ).columns.tolist()

            # First column contains the characteristic names
            characteristic_col = all_columns[0]
            geo_columns = all_columns[1:]

            # Read just the characteristics once
            with z.open(file) as f:
                characteristics = pd.read_csv(
                    f,
                    usecols=[characteristic_col],
                    encoding='latin1'
                )

            characteristics["CHARACTERISTIC_ID"] = np.arange(
                1,
                1 + len(characteristics)
            )

            col_chunk_size = 1000

            for i in range(0, len(geo_columns), col_chunk_size):
                current_geo_cols = geo_columns[i:i + col_chunk_size]

                # Reopen the file for every chunk
                with z.open(file) as f:
                    df = pd.read_csv(
                        f,
                        low_memory=False,
                        usecols=[characteristic_col] + current_geo_cols,
                        encoding='latin1',
                        na_values=['x', 'X', '..', 'F', '...', 'null', 'NULL']
                    )

                    # Convert:
                    #
                    # Characteristic | DA001 | DA002
                    # Population     | 100   | 200
                    # Male           | 50    | 100
                    #
                    # into:
                    #
                    # GEO_CODE | CHARACTERISTIC_NAME | VALUE
                    #
                    # Assign one characteristic ID to each row
                    df = df.melt(
                        id_vars=[characteristic_col],
                        var_name="GEO_CODE",
                        value_name="COUNT_TOTAL"
                    )

                    df["CHARACTERISTIC_ID"] = np.tile(
                        characteristics["CHARACTERISTIC_ID"].to_numpy(),
                        len(current_geo_cols)
                    )

                    df['GEO_CODE'] = df['GEO_CODE'].str[0:8]

                    df = df.rename(
                        columns={
                            characteristic_col: "CHARACTERISTIC_NAME"
                        }
                    )

                    df["COUNT_TOTAL"] = pd.to_numeric(
                        df["COUNT_TOTAL"],
                        errors="coerce"
                    ).astype(np.double)

                    schema = pa.schema([
                        ("GEO_CODE", pa.string()),
                        ("CHARACTERISTIC_ID", pa.int64()),
                        ("CHARACTERISTIC_NAME", pa.string()),
                        ("COUNT_TOTAL", pa.float64()),
                    ])                    

                    table = pa.Table.from_pandas(
                        df,
                        schema=schema,
                        preserve_index=False
                    )

                    if writer is None:
                        writer = pq.ParquetWriter(
                            output_path,
                            schema,
                            compression='snappy'
                        )

                    writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

In [ ]:
out_file = os.path.join(
    output_dir,
    "2011_raw_census_data.parquet"
)

out_file2 = os.path.join(
    output_dir,
    "2011_raw_nhs_data.parquet"
)

out_file3 = os.path.join(
    output_dir,
    "2011_raw_nhs_da.parquet"
)

process_2011_data(CENSUS_CONFIGS[2010], out_file, False)
process_2011_data(CENSUS_CONFIGS[2011], out_file2, True)
process_2011_da_data(out_file3)

# Function for processing XML files #

In [ ]:
def extract_value(df, concept):
    """
    Extract a value from generic:SeriesKey.generic:Value
    based on the concept attribute.

    Returns NULL if the concept does not exist.
    """
    return expr(f"""
        filter(
            `generic:SeriesKey`.`generic:Value`,
            x -> x._concept = '{concept}'
        )[0]._value
    """)

def get_xml_mappings(xml_mappings):
    mapping_df = pd.read_csv(
        xml_mappings
    )

    characteristic_mappings = {}

    for source_column in mapping_df.columns[1:]:
        mapping = {}

        for _, row in mapping_df.iterrows():
            canonical_id = row["Characteristic_ID"]
            source_id = row[source_column]

            if pd.notna(source_id):
                mapping[str(int(source_id))] = int(canonical_id)

        characteristic_mappings[source_column] = mapping

    return characteristic_mappings
    

def process_xml_data(spark, year_config, output_path):
    characteristic_mappings =  get_xml_mappings(year_config['xml_mappings'])
    with tempfile.TemporaryDirectory() as temp_dir:
        parquet_dir = os.path.join(temp_dir, "parquet")

        # Process each ZIP one XML file at a time.
        for zip_path in year_config["zips"]:
            print(f"Processing {zip_path}")

            # Get the XML filenames from the ZIP.
            result = subprocess.run(
                ["unzip", "-Z1", zip_path],
                capture_output=True,
                text=True,
                check=True,
            )

            xml_files = [
                name
                for name in result.stdout.splitlines()
                if name.lower().endswith(".xml")
                and not os.path.basename(name).startswith("Structure")
            ]

            print(f"Found {len(xml_files)} XML files")

            for i, xml_name in enumerate(xml_files, start=1):
                print(f"  [{i}/{len(xml_files)}] {xml_name}")

                xml_path = os.path.join(temp_dir, "input.xml")

                # Extract ONLY this XML file.
                with open(xml_path, "wb") as f:
                    subprocess.run(
                        ["unzip", "-p", zip_path, xml_name],
                        stdout=f,
                        check=True,
                    )

                # Let Spark process this XML.
                df = (
                    spark.read.format("xml")
                    .option("rowTag", "generic:Series")
                    .option("inferSchema", "false")
                    .load(f"file://{xml_path}")
                )

                result = df.select(
                    extract_value(df, "GEO").cast("string").alias("GEO_CODE"),
                    extract_value(df, "DIM0").alias("CHARACTERISTIC_ID"),
                    col("generic:Obs.generic:Time").cast("integer").alias("CENSUS_YEAR"),
                    col("generic:Obs.generic:ObsValue._value").cast("double").alias("COUNT_TOTAL"),
                )

                source_name = os.path.basename(xml_name)[8:-4]
                print(source_name)

                mapping = characteristic_mappings[source_name]

                mapping_expr = create_map(
                    *chain.from_iterable(
                        (lit(source_id), lit(canonical_id))
                        for source_id, canonical_id in mapping.items()
                    )
                )

                result = result.withColumn(
                    "CHARACTERISTIC_ID",
                    mapping_expr[col("CHARACTERISTIC_ID")]
                )

                # Append this XML's rows to the intermediate Parquet dataset.
                result.write \
                    .format("parquet") \
                    .mode("append") \
                    .option("compression", "snappy") \
                    .save(parquet_dir)

                # Remove the huge XML before moving to the next one.
                os.remove(xml_path)

        # Now combine the intermediate Parquet files into one file.
        print("Combining Parquet files...")

        df = spark.read.parquet(parquet_dir)

        temp_output = output_path + "_tmp"

        df.coalesce(1) \
            .write \
            .format("parquet") \
            .mode("overwrite") \
            .option("compression", "snappy") \
            .save(temp_output)

        # Find Spark's single output file.
        part_file = next(
            os.path.join(temp_output, name)
            for name in os.listdir(temp_output)
            if name.startswith("part-") and name.endswith(".parquet")
        )

        # Make sure the destination directory exists.
        os.makedirs(
            os.path.dirname(output_path) or ".",
            exist_ok=True,
        )

        # Move the actual Parquet file to the requested filename.
        if os.path.exists(output_path):
            os.remove(output_path)

        shutil.move(part_file, output_path)

        # Remove Spark's temporary output directory.
        shutil.rmtree(temp_output)

        print(f"Done: {output_path}")

# Preprocess 2006 Data #

In [ ]:
out_file = os.path.join(
    output_dir,
    "2006_raw_census_data.parquet"
)

spark = (
    SparkSession.builder
    .appName("2006 Census")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# this will take a very long time

process_xml_data(spark, CENSUS_CONFIGS[2006], out_file)

spark.stop()

# Preprocess 2001 Data #

In [ ]:
out_file = os.path.join(
    output_dir,
    "2001_raw_census_data.parquet"
)

spark = (
    SparkSession.builder
    .appName("2001 Census")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# this will take a very long time

process_xml_data(spark, CENSUS_CONFIGS[2001], out_file)

spark.stop()

# Preprocess 1996 Data #

In [ ]:
out_file = os.path.join(
    output_dir,
    "1996_raw_census_data.parquet"
)

spark = (
    SparkSession.builder
    .appName("1996 Census")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# this will take a very long time

process_xml_data(spark, CENSUS_CONFIGS[1996], out_file)

spark.stop()

# Process geographic information from XML files #

In [ ]:
def process_structure_xml(year_config, output_path, year):
    geographies = []

    with tempfile.TemporaryDirectory() as temp_dir:
        for zip_path in year_config["zips"]:
            print(f"Processing structure files in {zip_path}")

            # Get all XML filenames from the ZIP.
            result = subprocess.run(
                ["unzip", "-Z1", zip_path],
                capture_output=True,
                text=True,
                check=True,
            )

            structure_files = [
                name
                for name in result.stdout.splitlines()
                if os.path.basename(name).startswith("Structure")
                and name.lower().endswith(".xml")
            ]

            print(f"Found {len(structure_files)} Structure XML files")

            for i, structure_name in enumerate(structure_files, start=1):
                print(
                    f"  [{i}/{len(structure_files)}] "
                    f"{structure_name}"
                )

                xml_path = os.path.join(temp_dir, "structure.xml")

                # Extract this Structure XML.
                with open(xml_path, "wb") as f:
                    subprocess.run(
                        ["unzip", "-p", zip_path, structure_name],
                        stdout=f,
                        check=True,
                    )

                tree = ET.parse(xml_path)
                root = tree.getroot()

                namespaces = {
                    "structure": (
                        "http://www.SDMX.org/resources/"
                        "SDMXML/schemas/v2_0/structure"
                    )
                }

                # Find the geography code list.
                code_list = root.find(
                    ".//structure:CodeList[@id='CL_GEO']",
                    namespaces,
                )

                if code_list is None:
                    print("    No CL_GEO found")
                    continue

                for code in code_list.findall(
                    "structure:Code",
                    namespaces,
                ):
                    geo_code = code.get("value")

                    description = code.find(
                        "structure:Description",
                        namespaces,
                    )

                    if geo_code is None or description is None:
                        continue

                    geo_name = (description.text or "").strip()

                    geographies.append(
                        {
                            "GEO_CODE": geo_code,
                            "GEO_NAME": geo_name,
                        }
                    )

    # Combine all Structure files from all ZIPs.
    geo_df = pd.DataFrame(geographies)

    if geo_df.empty:
        print("No geographies found")
        return

    # A geography may occur in multiple Structure files.
    geo_df = geo_df.drop_duplicates(
        subset=["GEO_CODE"],
        keep="first",
    )

    # Make absolutely sure GEO_CODE remains a string.
    geo_df["GEO_CODE"] = geo_df["GEO_CODE"].astype(str)

    geo_df.to_parquet(
        output_path,
        index=False,
    )

    print(
        f"Wrote {len(geo_df)} geographies "
        f"to {output_path}"
    )

# Process 2006 Geographic information #

In [ ]:
out_file = os.path.join(
    second_out_dir,
    "2006_geographies.parquet"
)

process_structure_xml(CENSUS_CONFIGS[2006], out_file, "2006")

# Process 2001 Geographic information #

In [ ]:
out_file = os.path.join(
    second_out_dir,
    "2001_geographies.parquet"
)

process_structure_xml(CENSUS_CONFIGS[2001], out_file, "2001")

# Process 1996 Geographic information #

In [ ]:
out_file = os.path.join(
    second_out_dir,
    "1996_geographies.parquet"
)

process_structure_xml(CENSUS_CONFIGS[1996], out_file, "1996")

# Preprocess .csv files for 1981 and 1991 #

In [ ]:
def parse_geo_column(column):
    match = re.search(r'(.+?)\s*\(([^()]*)\)', column)

    if not match:
        return None, None

    geo_name = match.group(1).strip()
    geo_code = match.group(2).strip()

    return geo_code, geo_name


def clean_characteristic_name(name):
    return re.sub(r'^\d+\s+', '', str(name)).strip()


def get_csv_mappings(csv_mappings):
    mapping_df = pd.read_csv(csv_mappings)

    characteristic_mappings = {}

    for source_column in mapping_df.columns[1:]:
        mapping = {}

        for _, row in mapping_df.iterrows():
            canonical_id = row["Characteristic_ID"]
            source_id = row[source_column]

            if pd.notna(source_id):
                mapping[int(source_id)] = int(canonical_id)

        characteristic_mappings[source_column] = mapping

    return characteristic_mappings


def process_old_csv(year_config, output_path):
    characteristic_mappings = get_csv_mappings(
        year_config["csv_mappings"]
    )

    writer = None

    for zip_path in year_config["zips"]:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith(".csv"):
                    continue

                print(f"Processing {file}")

                with z.open(file) as f:
                    all_columns = pd.read_csv(
                        f,
                        nrows=0,
                        encoding="latin1",
                        na_values=[
                            "x", "X", "..", "F",
                            "...", "null", "NULL"
                        ],
                    ).columns.tolist()

                characteristic_col = all_columns[0]
                geo_columns = all_columns[1:]
                col_chunk_size = 1000

                source_name = file[:-4]

                if source_name not in characteristic_mappings:
                    print(
                        f"Skipping {file}: "
                        f"no mapping found for {source_name}"
                    )
                    continue

                mapping = characteristic_mappings[source_name]

                for i in range(0, len(geo_columns), col_chunk_size):
                    current_geo_cols = geo_columns[
                        i:i + col_chunk_size
                    ]

                    with z.open(file) as f:
                        df = pd.read_csv(
                            f,
                            low_memory=False,
                            usecols=[
                                characteristic_col
                            ] + current_geo_cols,
                            encoding="latin1",
                            na_values=[
                                "x", "X", "..", "F",
                                "...", "null", "NULL"
                            ],
                        )

                    # Rename the characteristic column
                    df.rename(
                        columns={
                            characteristic_col:
                                "CHARACTERISTIC_NAME"
                        },
                        inplace=True
                    )

                    # Source ID is the ROW NUMBER.
                    #
                    # First characteristic = 1
                    # Second characteristic = 2
                    # ...
                    df["SOURCE_ID"] = np.arange(1, len(df) + 1)

                    # Map source ID -> canonical ID
                    df["CHARACTERISTIC_ID"] = (
                        df["SOURCE_ID"].map(mapping)
                    )

                    # We no longer need SOURCE_ID
                    df.drop(
                        columns=["SOURCE_ID"],
                        inplace=True
                    )

                    # Convert from wide to long
                    df = df.melt(
                        id_vars=[
                            "CHARACTERISTIC_NAME",
                            "CHARACTERISTIC_ID"
                        ],
                        var_name="GEO_HEADER",
                        value_name="COUNT_TOTAL"
                    )

                    # Extract geography name/code
                    df["GEO_CODE"], df["GEO_NAME"] = zip(
                        *df["GEO_HEADER"].map(parse_geo_column)
                    )

                    df.drop(
                        columns=["GEO_HEADER"],
                        inplace=True
                    )

                    # Clean characteristic name
                    df["CHARACTERISTIC_NAME"] = (
                        df["CHARACTERISTIC_NAME"]
                        .map(clean_characteristic_name)
                    )

                    # Convert values
                    df["COUNT_TOTAL"] = pd.to_numeric(
                        df["COUNT_TOTAL"],
                        errors="coerce"
                    ).astype(np.float64)

                    # Remove characteristics that weren't in the mapping
                    df = df.dropna(
                        subset=["CHARACTERISTIC_ID"]
                    )

                    df["CHARACTERISTIC_ID"] = (
                        df["CHARACTERISTIC_ID"]
                        .astype(np.int64)
                    )

                    schema = pa.schema([
                        ("GEO_CODE", pa.string()),
                        ("GEO_NAME", pa.string()),
                        ("CHARACTERISTIC_ID", pa.int64()),
                        ("CHARACTERISTIC_NAME", pa.string()),
                        ("COUNT_TOTAL", pa.float64()),
                    ])

                    table = pa.Table.from_pandas(
                        df[
                            [
                                "GEO_CODE",
                                "GEO_NAME",
                                "CHARACTERISTIC_ID",
                                "CHARACTERISTIC_NAME",
                                "COUNT_TOTAL",
                            ]
                        ],
                        schema=schema,
                        preserve_index=False
                    )

                    if writer is None:
                        writer = pq.ParquetWriter(
                            output_path,
                            schema,
                            compression="snappy"
                        )

                    writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

# Preprocess 1991 Data #

In [ ]:
out_file = os.path.join(
    output_dir,
    "1991_raw_census_data.parquet"
)

process_old_csv(CENSUS_CONFIGS[1991], out_file)

# Preprocess 1981 Data #

In [ ]:
out_file = os.path.join(
    output_dir,
    "1981_raw_census_data.parquet"
)

process_old_csv(CENSUS_CONFIGS[1981], out_file)

# Preprocess 1986 Data #

In [ ]:
def parse_geo_column(column):
    match = re.search(r'(.+?)\s*\(([^()]*)\)', column)

    if not match:
        return None, None

    geo_name = match.group(1).strip()
    geo_code = match.group(2).strip()

    return geo_code, geo_name

def clean_characteristic_name(name):
    return re.sub(r'^\d+\s+', '', str(name)).strip()


def process_86_csv(year_config, output_path):
    writer = None

    # Number of characteristics in Part A.
    part_a_rows = 0

    # First determine the number of Part A characteristics.
    for zip_path in year_config["zips"]:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith(".csv"):
                    continue

                # Only Part A is used to determine the offset.
                if "part_a" not in file.lower():
                    continue

                with z.open(file) as f:
                    df = pd.read_csv(
                        f,
                        usecols=[0],
                        encoding="latin1",
                    )

                part_a_rows += len(df)
                break

    print(f"Part A characteristic count: {part_a_rows}")

    for zip_path in year_config["zips"]:
        if not os.path.exists(zip_path):
            print(f"Skipping missing file: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if not file.endswith(".csv"):
                    continue

                print(f"Processing {file}")

                with z.open(file) as f:
                    all_columns = pd.read_csv(
                        f,
                        nrows=0,
                        encoding="latin1",
                        na_values=[
                            "x", "X", "..", "F",
                            "...", "null", "NULL"
                        ],
                    ).columns.tolist()

                characteristic_col = all_columns[0]
                geo_columns = all_columns[1:]
                col_chunk_size = 1000

                # Determine whether this is Part A or Part B.
                is_part_b = "part_b" in file.lower()

                if is_part_b:
                    characteristic_offset = part_a_rows
                else:
                    characteristic_offset = 0

                for i in range(
                    0,
                    len(geo_columns),
                    col_chunk_size
                ):
                    current_geo_cols = geo_columns[
                        i:i + col_chunk_size
                    ]

                    with z.open(file) as f:
                        df = pd.read_csv(
                            f,
                            low_memory=False,
                            usecols=[
                                characteristic_col
                            ] + current_geo_cols,
                            encoding="latin1",
                            na_values=[
                                "x", "X", "..", "F",
                                "...", "null", "NULL"
                            ],
                        )

                    df.rename(
                        columns={
                            characteristic_col:
                                "CHARACTERISTIC_NAME"
                        },
                        inplace=True
                    )

                    # Characteristic ID is based purely on
                    # the row number.
                    df["CHARACTERISTIC_ID"] = (
                        np.arange(1, len(df) + 1)
                        + characteristic_offset
                    )

                    df = df.melt(
                        id_vars=[
                            "CHARACTERISTIC_NAME",
                            "CHARACTERISTIC_ID"
                        ],
                        var_name="GEO_HEADER",
                        value_name="COUNT_TOTAL"
                    )

                    df["GEO_CODE"], df["GEO_NAME"] = zip(
                        *df["GEO_HEADER"].map(parse_geo_column)
                    )

                    df.drop(
                        columns=["GEO_HEADER"],
                        inplace=True
                    )

                    df["CHARACTERISTIC_NAME"] = (
                        df["CHARACTERISTIC_NAME"]
                        .map(clean_characteristic_name)
                    )

                    df["COUNT_TOTAL"] = pd.to_numeric(
                        df["COUNT_TOTAL"],
                        errors="coerce"
                    ).astype(np.float64)

                    schema = pa.schema([
                        ("GEO_CODE", pa.string()),
                        ("GEO_NAME", pa.string()),
                        ("CHARACTERISTIC_ID", pa.int64()),
                        ("CHARACTERISTIC_NAME", pa.string()),
                        ("COUNT_TOTAL", pa.float64()),
                    ])

                    table = pa.Table.from_pandas(
                        df[
                            [
                                "GEO_CODE",
                                "GEO_NAME",
                                "CHARACTERISTIC_ID",
                                "CHARACTERISTIC_NAME",
                                "COUNT_TOTAL",
                            ]
                        ],
                        schema=schema,
                        preserve_index=False
                    )

                    if writer is None:
                        writer = pq.ParquetWriter(
                            output_path,
                            schema,
                            compression="snappy"
                        )

                    writer.write_table(table)

    if writer:
        writer.close()
        print(f"Finished writing: {output_path}")

In [ ]:
out_file = os.path.join(
    output_dir,
    "1986_raw_census_data.parquet"
)

process_86_csv(CENSUS_CONFIGS[1986], out_file)

Now, go run the second [jupyter notebook](./cleanup_parquet.ipynb)